## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 61 samples and 11666 columns.
First few rows of the data:


,"PFS time, months",Response status available,Sample_ID,RECIST Response,Primary_tumor._metastasis,Targeted therapeutics (next line),Age at biosampling,Gender,Tumor_site (biosampling),tissue_type,...,COL18A1,SLC19A1,COL6A1,COL6A2,LSS,MCM3AP,C21orf58,PCNT,DIP2A,PRMT2
0,6,1,OB_pat_LuC_104,PD,primary,"Pembrolizumab, Niraparib",45.0,male,Lung,Primary solid tumor,...,4.585306,2.262573,6.376399,6.027963,3.928306,5.213648,2.544580,4.829599,5.083457,4.750106
1,4,0,OB_pat_LuC_105,PR,primary,"Pembrolizumab, Niraparib",70.0,male,Lung,Primary solid tumor,...,4.501699,3.215302,5.298721,5.079625,4.306570,4.808402,2.096006,4.270306,4.983045,4.653635
2,3,1,OB_pat_LuC_107,PD,primary,"Nivolumab, Ipilimumab, Denosumab",60.0,male,Lung,Primary solid tumor,...,5.860539,3.635160,7.519625,7.193438,4.804173,5.193310,2.987562,4.998519,4.706178,4.395170
3,3,1,OB_pat_LuC_108,PD,primary,Pembrolizumab; Niraparib,66.0,male,Lung,Primary solid tumor,...,5.307713,3.002853,6.726389,6.701790,4.212309,5.200257,2.560028,4.790502,4.709332,4.455854
4,3,1,OB_pat_LuC_111,PD,primary,Pembrolizumab,65.0,male,Lung,Primary solid tumor,...,5.464326,2.294702,6.699775,6.314263,4.034288,5.120088,2.944485,5.197236,5.078670,4.688494


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
